# Lapsed Donor Reactivation Analysis

这是一个独立于 `Activist Codes / Tags` 治理项目的 fundraising 分析脚本。
目标是识别：
- 连续多年稳定捐赠
- 但在最近 5 年内停止捐赠
- 尤其是曾经达到 `$250+` 年捐赠额的人


In [ ]:
import csv
from html import escape
from pathlib import Path
import sqlite3
import nbformat as nbf


def find_repo_root():
    if "__file__" in globals():
        start = Path(__file__).resolve().parent
    else:
        start = Path.cwd().resolve()

    for candidate in [start, *start.parents]:
        if (candidate / "workbench" / "everyaction_workbench.sqlite").exists():
            return candidate

    raise FileNotFoundError("Could not locate the repository root from the current working directory.")


if "__file__" in globals():
    PROJECT_DIR = Path(__file__).resolve().parent
else:
    PROJECT_DIR = Path.cwd().resolve()

ROOT = find_repo_root()
DB_PATH = ROOT / "workbench" / "everyaction_workbench.sqlite"
REPORT_PATH = PROJECT_DIR / "lapsed_donor_reactivation_report.md"
EXECUTIVE_SUMMARY_REPORT_PATH = PROJECT_DIR / "fundraising_outreach_executive_summary.md"
EXECUTIVE_SUMMARY_NOTEBOOK_PATH = PROJECT_DIR / "fundraising_outreach_executive_summary.ipynb"
LAPSED_DONOR_EXPORT_PATH = PROJECT_DIR / "lapsed_consistent_donors.csv"
LAPSED_DONOR_250PLUS_EXPORT_PATH = PROJECT_DIR / "lapsed_consistent_donors_250plus.csv"
POSTCARD_OUTREACH_EXPORT_PATH = PROJECT_DIR / "people_added_since_january_postcard_outreach.csv"
FIGURE_DIR = PROJECT_DIR / "figures"
DONOR_VS_LEADERSHIP_COUNCIL_FIGURE_PATH = FIGURE_DIR / "donor_vs_leadership_council.svg"
MAILING_NO_EMAIL_FIGURE_PATH = FIGURE_DIR / "mailing_no_email_opportunity.svg"
POSTCARD_SINCE_JANUARY_FIGURE_PATH = FIGURE_DIR / "people_added_since_january_postcard_opportunity.svg"
LAPSED_DONOR_PRIORITY_FIGURE_PATH = FIGURE_DIR / "lapsed_donor_priority_tiers.svg"

ANALYSIS_AS_OF_DATE = "2026-04-16"
POSTCARD_OUTREACH_START_DATE = f"{ANALYSIS_AS_OF_DATE[:4]}-01-01"
MIN_CONSISTENT_GIVING_YEARS = 3
MIN_CONSECUTIVE_GIVING_STREAK = 3
HIGH_VALUE_ANNUAL_AMOUNT = 250

FIGURE_DIR.mkdir(parents=True, exist_ok=True)


def query_all(conn, sql, params=()):
    conn.row_factory = sqlite3.Row
    rows = conn.execute(sql, params).fetchall()
    return [dict(row) for row in rows]


def query_one(conn, sql, params=()):
    rows = query_all(conn, sql, params)
    return rows[0] if rows else {}


def markdown_table(rows, columns):
    if not rows:
        return "_No rows returned._"

    header = "| " + " | ".join(columns) + " |"
    separator = "| " + " | ".join("---" for _ in columns) + " |"
    body = []
    for row in rows:
        body.append("| " + " | ".join(str(row.get(column, "")) for column in columns) + " |")
    return "\n".join([header, separator, *body])


def write_csv_rows(path, rows, columns):
    with path.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=columns)
        writer.writeheader()
        for row in rows:
            writer.writerow({column: row.get(column, "") for column in columns})


def write_svg_bar_chart(path, title, items, width=980, height=420, bar_color="#0f766e"):
    if not items:
        return

    left = 80
    right = 40
    top = 70
    bottom = 90
    plot_width = width - left - right
    plot_height = height - top - bottom
    max_value = max(value for _label, value in items) or 1
    gap = 18
    bar_width = max(22, int((plot_width - gap * (len(items) - 1)) / max(1, len(items))))

    parts = [
        f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}">',
        '<style>text { font-family: Arial, sans-serif; fill: #222; } .small { font-size: 12px; } .label { font-size: 12px; } .title { font-size: 20px; font-weight: bold; }</style>',
        f'<rect x="0" y="0" width="{width}" height="{height}" fill="#ffffff"/>',
        f'<text x="{width/2}" y="30" text-anchor="middle" class="title">{escape(title)}</text>',
        f'<line x1="{left}" y1="{top + plot_height}" x2="{left + plot_width}" y2="{top + plot_height}" stroke="#444" stroke-width="1"/>',
        f'<line x1="{left}" y1="{top}" x2="{left}" y2="{top + plot_height}" stroke="#444" stroke-width="1"/>',
    ]

    for tick in range(5):
        value = max_value * tick / 4
        y = top + plot_height - (plot_height * tick / 4)
        parts.append(f'<line x1="{left - 4}" y1="{y}" x2="{left + plot_width}" y2="{y}" stroke="#e9e9e9" stroke-width="1"/>')
        parts.append(f'<text x="{left - 10}" y="{y + 4}" text-anchor="end" class="small">{int(round(value))}</text>')

    x = left
    for label, value in items:
        bar_height = 0 if max_value == 0 else plot_height * value / max_value
        y = top + plot_height - bar_height
        parts.append(f'<rect x="{x}" y="{y}" width="{bar_width}" height="{bar_height}" fill="{bar_color}" rx="4"/>')
        parts.append(f'<text x="{x + bar_width/2}" y="{y - 8}" text-anchor="middle" class="small">{value}</text>')
        parts.append(f'<text x="{x + bar_width/2}" y="{top + plot_height + 18}" text-anchor="middle" class="label">{escape(str(label))}</text>')
        x += bar_width + gap

    parts.append("</svg>")
    path.write_text("\n".join(parts), encoding="utf-8")


def write_svg_horizontal_bar_chart(path, title, items, width=980, bar_color="#0f766e"):
    if not items:
        return

    left = 250
    right = 110
    top = 70
    row_height = 54
    bottom = 36
    height = top + bottom + row_height * len(items)
    plot_width = width - left - right
    max_value = max(value for _label, value in items) or 1

    parts = [
        f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}">',
        '<style>text { font-family: Arial, sans-serif; fill: #222; } .small { font-size: 12px; } .label { font-size: 14px; } .title { font-size: 20px; font-weight: bold; }</style>',
        f'<rect x="0" y="0" width="{width}" height="{height}" fill="#ffffff"/>',
        f'<text x="{width/2}" y="32" text-anchor="middle" class="title">{escape(title)}</text>',
    ]

    for index, (label, value) in enumerate(items):
        y = top + index * row_height
        bar_y = y - 14
        bar_width = 0 if max_value == 0 else plot_width * value / max_value
        parts.append(f'<text x="{left - 12}" y="{y + 2}" text-anchor="end" class="label">{escape(str(label))}</text>')
        parts.append(f'<rect x="{left}" y="{bar_y}" width="{plot_width}" height="22" fill="#f1f5f9" rx="4"/>')
        parts.append(f'<rect x="{left}" y="{bar_y}" width="{bar_width}" height="22" fill="{bar_color}" rx="4"/>')
        value_x = min(left + bar_width + 10, width - right + 10)
        parts.append(f'<text x="{value_x}" y="{y + 2}" text-anchor="start" class="small">{value:,}</text>')

    parts.append("</svg>")
    path.write_text("\n".join(parts), encoding="utf-8")


## 1. 数据来源与口径

使用表：
- `all_contributions_ever`

这里采用保守口径：
- 只保留看起来像个人姓名的记录：`contact_name` 形如 `Last, First`
- `consistent` = 至少 3 个不同捐赠年份，且最长连续捐赠 streak 至少 3 年
- `lapsed` = 截至 `2026-04-16`，最近一笔捐赠发生在 1 到 5 年前
- `priority_250_plus` = 至少有 1 个自然年的年捐赠额达到 `$250+`


In [ ]:
conn = sqlite3.connect(DB_PATH)

data_check = query_one(
    conn,
    """
    SELECT
      COUNT(*) AS contribution_rows,
      COUNT(DISTINCT vanid) AS distinct_vanids,
      COUNT(DISTINCT contact_name) AS distinct_contact_names,
      MIN(date_received) AS first_gift_date,
      MAX(date_received) AS latest_gift_date
    FROM all_contributions_ever
    """
)

print(data_check)


## 1A. Donor universe 与 Leadership Council 口径

为了支持总量对比，这里额外计算：
- 全部 donor 去重人数
- 符合当前分析 person-like 规则的 donor 去重人数
- 通过 `19LCPin` / `20LCPin` 推断出的 Leadership Council 人数


In [ ]:
donor_vs_leadership_council_summary = query_one(
    conn,
    """
    WITH person_like_donors AS (
      SELECT DISTINCT vanid
      FROM all_contributions_ever
      WHERE contact_name LIKE '%, %'
    ), donor_universe AS (
      SELECT
        COUNT(DISTINCT vanid) AS total_donors,
        COUNT(DISTINCT CASE WHEN contact_name LIKE '%, %' THEN vanid END) AS person_like_donors
      FROM all_contributions_ever
    ), leadership_council AS (
      SELECT
        COUNT(DISTINCT vanid) AS total_lc_members,
        COUNT(DISTINCT CASE WHEN vanid IN (SELECT vanid FROM person_like_donors) THEN vanid END) AS lc_members_in_person_like_donor_base
      FROM all_activist_codes_applied
      WHERE activistcodename IN ('19LCPin', '20LCPin')
    )
    SELECT
      donor_universe.total_donors,
      donor_universe.person_like_donors,
      leadership_council.total_lc_members,
      leadership_council.lc_members_in_person_like_donor_base,
      leadership_council.total_lc_members - leadership_council.lc_members_in_person_like_donor_base AS lc_members_outside_person_like_donor_base,
      ROUND(
        100.0 * leadership_council.lc_members_in_person_like_donor_base / NULLIF(donor_universe.person_like_donors, 0),
        2
      ) AS lc_share_of_person_like_donors
    FROM donor_universe
    CROSS JOIN leadership_council
    """
)

write_svg_bar_chart(
    DONOR_VS_LEADERSHIP_COUNCIL_FIGURE_PATH,
    "Donors vs Leadership Council Members",
    [
        ("All donors", donor_vs_leadership_council_summary["total_donors"]),
        ("LC members", donor_vs_leadership_council_summary["total_lc_members"]),
    ],
)

print(donor_vs_leadership_council_summary)
print(DONOR_VS_LEADERSHIP_COUNCIL_FIGURE_PATH)


## 1B. 数据库里“有邮寄地址但没有邮箱”的人数

这一步直接回答一个 outreach 相关问题：
- 数据库里有多少 `Person` 记录可以寄信
- 但当前没有任何邮箱地址

口径定义：
- 主表使用 `database_list`，因为它是一条记录对应一个联系人
- `mailing address` = `address + city + state_province + zip_postal` 都非空
- `no email` = `preferredemail` 和 `otheremail` 都为空
- 额外交叉校验 `addresses_report` 中 `Current + complete address` 的人数


In [ ]:
database_mail_without_email_summary = query_one(
    conn,
    """
    WITH person_base AS (
      SELECT
        vanid,
        CASE
          WHEN TRIM(COALESCE(address, '')) <> ''
           AND TRIM(COALESCE(city, '')) <> ''
           AND TRIM(COALESCE(state_province, '')) <> ''
           AND TRIM(COALESCE(zip_postal, '')) <> ''
          THEN 1 ELSE 0
        END AS has_complete_mailing_address,
        CASE
          WHEN TRIM(COALESCE(preferredemail, '')) <> ''
            OR TRIM(COALESCE(otheremail, '')) <> ''
          THEN 1 ELSE 0
        END AS has_any_email,
        CAST(COALESCE(nomail, '0') AS INT) AS nomail
      FROM database_list
      WHERE type_of_contact = 'Person'
    )
    SELECT
      COUNT(*) AS total_people,
      SUM(has_complete_mailing_address) AS people_with_complete_mailing_address,
      SUM(CASE WHEN has_any_email = 0 THEN 1 ELSE 0 END) AS people_with_no_email,
      SUM(CASE WHEN has_complete_mailing_address = 1 AND has_any_email = 0 THEN 1 ELSE 0 END) AS mailing_address_but_no_email,
      ROUND(
        100.0 * SUM(CASE WHEN has_complete_mailing_address = 1 AND has_any_email = 0 THEN 1 ELSE 0 END) / COUNT(*),
        2
      ) AS share_of_all_people,
      ROUND(
        100.0 * SUM(CASE WHEN has_complete_mailing_address = 1 AND has_any_email = 0 THEN 1 ELSE 0 END)
        / NULLIF(SUM(has_complete_mailing_address), 0),
        2
      ) AS share_of_people_with_mailing_address,
      SUM(
        CASE
          WHEN has_complete_mailing_address = 1 AND has_any_email = 0 AND nomail = 0
          THEN 1 ELSE 0
        END
      ) AS mailing_address_but_no_email_and_mailable
    FROM person_base
    """
)

database_mail_without_email_crosscheck = query_one(
    conn,
    """
    WITH current_complete_addresses AS (
      SELECT DISTINCT vanid
      FROM addresses_report
      WHERE status = 'Current'
        AND is_complete_address = '1'
    ), person_base AS (
      SELECT
        vanid,
        CASE
          WHEN TRIM(COALESCE(preferredemail, '')) <> ''
            OR TRIM(COALESCE(otheremail, '')) <> ''
          THEN 1 ELSE 0
        END AS has_any_email
      FROM database_list
      WHERE type_of_contact = 'Person'
    )
    SELECT COUNT(*) AS current_complete_address_no_email
    FROM current_complete_addresses a
    JOIN person_base p USING (vanid)
    WHERE p.has_any_email = 0
    """
)

print(
    markdown_table(
        [database_mail_without_email_summary],
        [
            "total_people",
            "people_with_complete_mailing_address",
            "people_with_no_email",
            "mailing_address_but_no_email",
            "share_of_all_people",
            "share_of_people_with_mailing_address",
            "mailing_address_but_no_email_and_mailable",
        ],
    )
)
print()
print(
    "Interpretation: "
    f"{database_mail_without_email_summary['mailing_address_but_no_email']:,} people have a complete mailing address "
    "but no email on file. "
    f"That is {database_mail_without_email_summary['share_of_all_people']}% of all `Person` records and "
    f"{database_mail_without_email_summary['share_of_people_with_mailing_address']}% of people with a complete "
    "mailing address. "
    f"{database_mail_without_email_summary['mailing_address_but_no_email_and_mailable']:,} of them are not flagged "
    "`nomail`, so they remain reachable by direct mail. "
    f"A cross-check against `addresses_report` returns "
    f"{database_mail_without_email_crosscheck['current_complete_address_no_email']:,}, only "
    f"{database_mail_without_email_crosscheck['current_complete_address_no_email'] - database_mail_without_email_summary['mailing_address_but_no_email']:,} "
    "higher than the `database_list` result."
)


## 1C. 识别自今年 1 月以来新增、可寄 postcard、但没有邮箱的人

这里把用户问题收窄成一个可执行名单：
- `added since January` = `date_created >= 2026-01-01`
- `possible postcard outreach` = 地址完整、没有邮箱、且 `nomail = 0`
- 额外保留 `origincodename` 方便回看这些联系人从什么来源进入数据库


In [ ]:
postcard_outreach_since_january_summary = query_one(
    conn,
    f"""
    WITH person_base AS (
      SELECT
        vanid,
        last,
        first,
        mid,
        suf,
        address,
        city,
        state_province,
        zip_postal,
        date_created,
        date(substr(date_acquired, 7, 4) || '-' || substr(date_acquired, 1, 2) || '-' || substr(date_acquired, 4, 2)) AS date_acquired_iso,
        origincodename,
        CAST(COALESCE(noemail, '0') AS INT) AS noemail_flag,
        CAST(COALESCE(nomail, '0') AS INT) AS nomail,
        CASE
          WHEN TRIM(COALESCE(address, '')) <> ''
           AND TRIM(COALESCE(city, '')) <> ''
           AND TRIM(COALESCE(state_province, '')) <> ''
           AND TRIM(COALESCE(zip_postal, '')) <> ''
          THEN 1 ELSE 0
        END AS has_complete_mailing_address,
        CASE
          WHEN TRIM(COALESCE(preferredemail, '')) <> ''
            OR TRIM(COALESCE(otheremail, '')) <> ''
          THEN 1 ELSE 0
        END AS has_any_email
      FROM database_list
      WHERE type_of_contact = 'Person'
    )
    SELECT
      COUNT(*) AS people_added_since_january,
      SUM(has_complete_mailing_address) AS people_added_since_january_with_complete_address,
      SUM(CASE WHEN has_any_email = 0 THEN 1 ELSE 0 END) AS people_added_since_january_with_no_email,
      SUM(CASE WHEN has_complete_mailing_address = 1 AND has_any_email = 0 THEN 1 ELSE 0 END) AS people_added_since_january_with_address_but_no_email,
      SUM(
        CASE
          WHEN has_complete_mailing_address = 1 AND has_any_email = 0 AND nomail = 0
          THEN 1 ELSE 0
        END
      ) AS people_added_since_january_postcard_candidates,
      ROUND(
        100.0 * SUM(CASE WHEN has_complete_mailing_address = 1 AND has_any_email = 0 AND nomail = 0 THEN 1 ELSE 0 END)
        / COUNT(*),
        2
      ) AS share_of_people_added_since_january,
      ROUND(
        100.0 * SUM(CASE WHEN has_complete_mailing_address = 1 AND has_any_email = 0 AND nomail = 0 THEN 1 ELSE 0 END)
        / NULLIF(SUM(has_complete_mailing_address), 0),
        2
      ) AS share_of_people_added_since_january_with_complete_address
    FROM person_base
    WHERE date(date_created) >= date('{POSTCARD_OUTREACH_START_DATE}')
    """
)

postcard_outreach_since_january_rows = query_all(
    conn,
    f"""
    WITH person_base AS (
      SELECT
        vanid,
        TRIM(
          COALESCE(last, '')
          || CASE WHEN TRIM(COALESCE(first, '')) <> '' THEN ', ' || TRIM(first) ELSE '' END
          || CASE WHEN TRIM(COALESCE(mid, '')) <> '' THEN ' ' || TRIM(mid) ELSE '' END
          || CASE WHEN TRIM(COALESCE(suf, '')) <> '' THEN ' ' || TRIM(suf) ELSE '' END
        ) AS contact_name,
        address,
        city,
        state_province,
        zip_postal,
        date_created,
        date(substr(date_acquired, 7, 4) || '-' || substr(date_acquired, 1, 2) || '-' || substr(date_acquired, 4, 2)) AS date_acquired_iso,
        origincodename,
        CAST(COALESCE(noemail, '0') AS INT) AS noemail_flag,
        CAST(COALESCE(nomail, '0') AS INT) AS nomail,
        CASE
          WHEN TRIM(COALESCE(address, '')) <> ''
           AND TRIM(COALESCE(city, '')) <> ''
           AND TRIM(COALESCE(state_province, '')) <> ''
           AND TRIM(COALESCE(zip_postal, '')) <> ''
          THEN 1 ELSE 0
        END AS has_complete_mailing_address,
        CASE
          WHEN TRIM(COALESCE(preferredemail, '')) <> ''
            OR TRIM(COALESCE(otheremail, '')) <> ''
          THEN 1 ELSE 0
        END AS has_any_email
      FROM database_list
      WHERE type_of_contact = 'Person'
    )
    SELECT
      vanid,
      contact_name,
      address,
      city,
      state_province,
      zip_postal,
      date_created,
      date_acquired_iso,
      origincodename,
      noemail_flag,
      nomail
    FROM person_base
    WHERE date(date_created) >= date('{POSTCARD_OUTREACH_START_DATE}')
      AND has_complete_mailing_address = 1
      AND has_any_email = 0
      AND nomail = 0
    ORDER BY date_created DESC, contact_name
    """
)

print(
    markdown_table(
        [postcard_outreach_since_january_summary],
        [
            "people_added_since_january",
            "people_added_since_january_with_complete_address",
            "people_added_since_january_with_no_email",
            "people_added_since_january_with_address_but_no_email",
            "people_added_since_january_postcard_candidates",
            "share_of_people_added_since_january",
            "share_of_people_added_since_january_with_complete_address",
        ],
    )
)
print()
print(
    "Interpretation: "
    f"using `date_created >= {POSTCARD_OUTREACH_START_DATE}`, "
    f"{postcard_outreach_since_january_summary['people_added_since_january_postcard_candidates']:,} people added "
    "since January have a complete mailing address, no email on file, and are not flagged `nomail`. "
    f"That is {postcard_outreach_since_january_summary['share_of_people_added_since_january']}% of all people added "
    "since January and "
    f"{postcard_outreach_since_january_summary['share_of_people_added_since_january_with_complete_address']}% of "
    "those with a complete address."
)
print()
print(
    markdown_table(
        postcard_outreach_since_january_rows,
        [
            "vanid",
            "contact_name",
            "city",
            "state_province",
            "zip_postal",
            "date_created",
            "origincodename",
        ],
    )
)

write_svg_horizontal_bar_chart(
    MAILING_NO_EMAIL_FIGURE_PATH,
    "Direct-Mail Opportunity in the Database",
    [
        ("All people", database_mail_without_email_summary["total_people"]),
        ("With complete address", database_mail_without_email_summary["people_with_complete_mailing_address"]),
        ("With no email", database_mail_without_email_summary["people_with_no_email"]),
        (
            "Mailable address + no email",
            database_mail_without_email_summary["mailing_address_but_no_email_and_mailable"],
        ),
    ],
    bar_color="#c2410c",
)

write_svg_horizontal_bar_chart(
    POSTCARD_SINCE_JANUARY_FIGURE_PATH,
    f"People Added Since {POSTCARD_OUTREACH_START_DATE}",
    [
        ("People added since Jan 1", postcard_outreach_since_january_summary["people_added_since_january"]),
        (
            "With complete address",
            postcard_outreach_since_january_summary["people_added_since_january_with_complete_address"],
        ),
        ("With no email", postcard_outreach_since_january_summary["people_added_since_january_with_no_email"]),
        (
            "Postcard candidates",
            postcard_outreach_since_january_summary["people_added_since_january_postcard_candidates"],
        ),
    ],
    bar_color="#0f766e",
)


## 2. 建立 donor rollup 逻辑

先按 `vanid + 年份` 聚合，再计算每个 donor 的：
- giving years
- longest consecutive streak
- last gift date
- lifetime amount
- max annual amount
- `$250+` 年份数量


In [ ]:
lapsed_donor_base_sql = f"""
WITH raw_contributions AS (
  SELECT
    vanid,
    contact_name,
    date_received,
    CAST(amount AS REAL) AS amount_value,
    CAST(substr(date_received, 1, 4) AS INT) AS gift_year
  FROM all_contributions_ever
  WHERE contact_name LIKE '%, %'
), donor_years AS (
  SELECT
    vanid,
    MAX(contact_name) AS contact_name,
    gift_year,
    SUM(amount_value) AS amount_in_year,
    COUNT(*) AS gifts_in_year,
    MAX(date_received) AS last_gift_in_year,
    MAX(amount_value) AS max_single_gift_in_year
  FROM raw_contributions
  GROUP BY vanid, gift_year
), donor_rollup AS (
  SELECT
    vanid,
    MAX(contact_name) AS contact_name,
    COUNT(*) AS giving_years,
    MIN(gift_year) AS first_gift_year,
    MAX(gift_year) AS last_gift_year,
    SUM(amount_in_year) AS lifetime_amount,
    AVG(amount_in_year) AS avg_annual_amount,
    MAX(amount_in_year) AS max_annual_amount,
    MAX(max_single_gift_in_year) AS max_single_gift_amount,
    SUM(gifts_in_year) AS lifetime_gifts,
    SUM(CASE WHEN amount_in_year >= {HIGH_VALUE_ANNUAL_AMOUNT} THEN 1 ELSE 0 END) AS years_at_250_plus,
    MAX(last_gift_in_year) AS last_gift_date
  FROM donor_years
  GROUP BY vanid
), ordered_years AS (
  SELECT
    vanid,
    gift_year,
    gift_year - ROW_NUMBER() OVER (PARTITION BY vanid ORDER BY gift_year) AS streak_group
  FROM donor_years
), streaks AS (
  SELECT
    vanid,
    MIN(gift_year) AS streak_start_year,
    MAX(gift_year) AS streak_end_year,
    COUNT(*) AS streak_years
  FROM ordered_years
  GROUP BY vanid, streak_group
), longest_streaks AS (
  SELECT
    vanid,
    MAX(streak_years) AS longest_streak
  FROM streaks
  GROUP BY vanid
), candidates AS (
  SELECT
    r.vanid,
    r.contact_name,
    r.giving_years,
    COALESCE(l.longest_streak, 1) AS longest_streak,
    r.first_gift_year,
    r.last_gift_year,
    r.last_gift_date,
    CAST(julianday('{ANALYSIS_AS_OF_DATE}') - julianday(r.last_gift_date) AS INT) AS days_since_last_gift,
    ROUND(r.lifetime_amount, 2) AS lifetime_amount,
    ROUND(r.avg_annual_amount, 2) AS avg_annual_amount,
    ROUND(r.max_annual_amount, 2) AS max_annual_amount,
    ROUND(r.max_single_gift_amount, 2) AS max_single_gift_amount,
    r.lifetime_gifts,
    r.years_at_250_plus,
    CASE WHEN r.years_at_250_plus >= 1 THEN 1 ELSE 0 END AS priority_250_plus,
    CASE
      WHEN r.years_at_250_plus >= 1 AND date(r.last_gift_date) >= date('{ANALYSIS_AS_OF_DATE}', '-2 years') THEN 'Tier 1'
      WHEN r.years_at_250_plus >= 1 THEN 'Tier 2'
      WHEN date(r.last_gift_date) >= date('{ANALYSIS_AS_OF_DATE}', '-2 years') THEN 'Tier 3'
      ELSE 'Tier 4'
    END AS recapture_priority
  FROM donor_rollup r
  LEFT JOIN longest_streaks l USING (vanid)
  WHERE r.giving_years >= {MIN_CONSISTENT_GIVING_YEARS}
    AND COALESCE(l.longest_streak, 1) >= {MIN_CONSECUTIVE_GIVING_STREAK}
    AND date(r.last_gift_date) <= date('{ANALYSIS_AS_OF_DATE}', '-1 year')
    AND date(r.last_gift_date) > date('{ANALYSIS_AS_OF_DATE}', '-5 years')
)
"""


## 3. 输出摘要与优先级结果


In [ ]:
lapsed_donor_summary = query_one(
    conn,
    f"""
    {lapsed_donor_base_sql}
    SELECT
      COUNT(*) AS candidate_count,
      SUM(priority_250_plus) AS candidate_250plus_count,
      SUM(CASE WHEN recapture_priority = 'Tier 1' THEN 1 ELSE 0 END) AS tier_1_count,
      SUM(CASE WHEN recapture_priority = 'Tier 2' THEN 1 ELSE 0 END) AS tier_2_count,
      SUM(CASE WHEN recapture_priority = 'Tier 3' THEN 1 ELSE 0 END) AS tier_3_count,
      SUM(CASE WHEN recapture_priority = 'Tier 4' THEN 1 ELSE 0 END) AS tier_4_count,
      ROUND(AVG(giving_years), 2) AS avg_giving_years,
      ROUND(AVG(longest_streak), 2) AS avg_longest_streak,
      ROUND(AVG(lifetime_amount), 2) AS avg_lifetime_amount,
      ROUND(AVG(max_annual_amount), 2) AS avg_max_annual_amount
    FROM candidates
    """
)

lapsed_donor_last_gift_year_rows = query_all(
    conn,
    f"""
    {lapsed_donor_base_sql}
    SELECT
      last_gift_year,
      COUNT(*) AS candidate_count,
      SUM(priority_250_plus) AS candidate_250plus_count
    FROM candidates
    GROUP BY last_gift_year
    ORDER BY last_gift_year DESC
    """
)

lapsed_donor_top_rows = query_all(
    conn,
    f"""
    {lapsed_donor_base_sql}
    SELECT
      vanid,
      contact_name,
      recapture_priority,
      giving_years,
      longest_streak,
      first_gift_year,
      last_gift_year,
      last_gift_date,
      days_since_last_gift,
      lifetime_amount,
      avg_annual_amount,
      max_annual_amount,
      max_single_gift_amount,
      years_at_250_plus
    FROM candidates
    ORDER BY priority_250_plus DESC, last_gift_date DESC, years_at_250_plus DESC, max_annual_amount DESC, lifetime_amount DESC
    LIMIT 20
    """
)

print(lapsed_donor_summary)
print()
print(markdown_table(lapsed_donor_last_gift_year_rows, ["last_gift_year", "candidate_count", "candidate_250plus_count"]))
print()
print(
    markdown_table(
        lapsed_donor_top_rows,
        [
            "vanid",
            "contact_name",
            "recapture_priority",
            "giving_years",
            "longest_streak",
            "last_gift_date",
            "days_since_last_gift",
            "lifetime_amount",
            "max_annual_amount",
            "years_at_250_plus",
        ],
    )
)

write_svg_bar_chart(
    LAPSED_DONOR_PRIORITY_FIGURE_PATH,
    "Lapsed Donor Reactivation Priority Tiers",
    [
        ("Tier 1", lapsed_donor_summary["tier_1_count"]),
        ("Tier 2", lapsed_donor_summary["tier_2_count"]),
        ("Tier 3", lapsed_donor_summary["tier_3_count"]),
        ("Tier 4", lapsed_donor_summary["tier_4_count"]),
    ],
    bar_color="#b91c1c",
)


## 4. Donor universe vs Leadership Council


In [ ]:
print(
    markdown_table(
        [donor_vs_leadership_council_summary],
        [
            "total_donors",
            "person_like_donors",
            "total_lc_members",
            "lc_members_in_person_like_donor_base",
            "lc_members_outside_person_like_donor_base",
            "lc_share_of_person_like_donors",
        ],
    )
)


## 5. 导出 CSV 名单


In [ ]:
lapsed_donor_export_columns = [
    "vanid",
    "contact_name",
    "recapture_priority",
    "priority_250_plus",
    "giving_years",
    "longest_streak",
    "first_gift_year",
    "last_gift_year",
    "last_gift_date",
    "days_since_last_gift",
    "lifetime_gifts",
    "lifetime_amount",
    "avg_annual_amount",
    "max_annual_amount",
    "max_single_gift_amount",
    "years_at_250_plus",
]

lapsed_donor_export_rows = query_all(
    conn,
    f"""
    {lapsed_donor_base_sql}
    SELECT
      vanid,
      contact_name,
      recapture_priority,
      priority_250_plus,
      giving_years,
      longest_streak,
      first_gift_year,
      last_gift_year,
      last_gift_date,
      days_since_last_gift,
      lifetime_gifts,
      lifetime_amount,
      avg_annual_amount,
      max_annual_amount,
      max_single_gift_amount,
      years_at_250_plus
    FROM candidates
    ORDER BY priority_250_plus DESC, last_gift_date DESC, years_at_250_plus DESC, max_annual_amount DESC, lifetime_amount DESC
    """
)

lapsed_donor_250plus_rows = [row for row in lapsed_donor_export_rows if row["priority_250_plus"] == 1]

postcard_outreach_export_columns = [
    "vanid",
    "contact_name",
    "address",
    "city",
    "state_province",
    "zip_postal",
    "date_created",
    "date_acquired_iso",
    "origincodename",
    "noemail_flag",
    "nomail",
]

write_csv_rows(LAPSED_DONOR_EXPORT_PATH, lapsed_donor_export_rows, lapsed_donor_export_columns)
write_csv_rows(LAPSED_DONOR_250PLUS_EXPORT_PATH, lapsed_donor_250plus_rows, lapsed_donor_export_columns)
write_csv_rows(
    POSTCARD_OUTREACH_EXPORT_PATH,
    postcard_outreach_since_january_rows,
    postcard_outreach_export_columns,
)

print(LAPSED_DONOR_EXPORT_PATH)
print(LAPSED_DONOR_250PLUS_EXPORT_PATH)
print(POSTCARD_OUTREACH_EXPORT_PATH)

tier_1_and_2_count = lapsed_donor_summary["tier_1_count"] + lapsed_donor_summary["tier_2_count"]
recent_lapsed_2024_2025_count = sum(
    row["candidate_count"] for row in lapsed_donor_last_gift_year_rows if row["last_gift_year"] in (2024, 2025)
)
postcard_candidates_in_arizona = sum(
    1 for row in postcard_outreach_since_january_rows if row["state_province"] == "AZ"
)
postcard_candidates_outside_arizona = (
    postcard_outreach_since_january_summary["people_added_since_january_postcard_candidates"]
    - postcard_candidates_in_arizona
)

executive_summary_rows = [
    {
        "question": "1. How many people are reachable by mail but missing an email address?",
        "result": f"{database_mail_without_email_summary['mailing_address_but_no_email_and_mailable']:,} people",
        "recommended_action": "Create a dedicated direct-mail, postcard, and email-acquisition segment.",
    },
    {
        "question": "2. Which people added since January are good postcard candidates?",
        "result": f"{postcard_outreach_since_january_summary['people_added_since_january_postcard_candidates']:,} people",
        "recommended_action": "Run a small welcome or reactivation postcard test immediately.",
    },
    {
        "question": "3. How large is Leadership Council within the donor base?",
        "result": (
            f"{donor_vs_leadership_council_summary['total_lc_members']:,} people, "
            "equal to "
            f"{donor_vs_leadership_council_summary['lc_share_of_person_like_donors']}%"
        ),
        "recommended_action": "Treat LC as a distinct stewardship segment, not a proxy for the general donor base.",
    },
    {
        "question": "4. Which lapsed donors should be prioritized for reactivation?",
        "result": (
            f"{tier_1_and_2_count:,} high-priority people, including "
            f"{lapsed_donor_summary['candidate_250plus_count']:,} with a $250+ giving history"
        ),
        "recommended_action": "Start with Tier 1 and Tier 2, then expand to Tier 3 and Tier 4.",
    },
]


## 6. 输出 markdown 报告


In [ ]:
report_lines = [
    "# Lapsed Donor Reactivation Analysis",
    "",
    "## Data Check",
    "",
    markdown_table(
        [data_check],
        ["contribution_rows", "distinct_vanids", "distinct_contact_names", "first_gift_date", "latest_gift_date"],
    ),
    "",
    "## Rules Used",
    "",
    (
        f"As of {ANALYSIS_AS_OF_DATE}: keep person-like names only (`Last, First`), require at least "
        f"{MIN_CONSISTENT_GIVING_YEARS} giving years and a longest consecutive streak of at least "
        f"{MIN_CONSECUTIVE_GIVING_STREAK} years, and define `priority_250_plus` as at least one giving year at "
        f"${HIGH_VALUE_ANNUAL_AMOUNT}+."
    ),
    "",
    "## Mailing Address but No Email",
    "",
    markdown_table(
        [database_mail_without_email_summary],
        [
            "total_people",
            "people_with_complete_mailing_address",
            "people_with_no_email",
            "mailing_address_but_no_email",
            "share_of_all_people",
            "share_of_people_with_mailing_address",
            "mailing_address_but_no_email_and_mailable",
        ],
    ),
    "",
    (
        f"Interpretation: {database_mail_without_email_summary['mailing_address_but_no_email']:,} `Person` records "
        "have a complete mailing address but no email on file. "
        f"That is {database_mail_without_email_summary['share_of_all_people']}% of all people and "
        f"{database_mail_without_email_summary['share_of_people_with_mailing_address']}% of people with a complete "
        "mailing address. "
        f"{database_mail_without_email_summary['mailing_address_but_no_email_and_mailable']:,} of those records are "
        "not flagged `nomail`, so they remain usable for direct-mail outreach. "
        f"A cross-check against `addresses_report` returns "
        f"{database_mail_without_email_crosscheck['current_complete_address_no_email']:,}, only "
        f"{database_mail_without_email_crosscheck['current_complete_address_no_email'] - database_mail_without_email_summary['mailing_address_but_no_email']:,} "
        "higher than the `database_list` result."
    ),
    "",
    "## People Added Since January for Possible Postcard Outreach",
    "",
    (
        f"Assumption used: `since January` means `date_created >= {POSTCARD_OUTREACH_START_DATE}` "
        f"(that is, since {POSTCARD_OUTREACH_START_DATE})."
    ),
    "",
    markdown_table(
        [postcard_outreach_since_january_summary],
        [
            "people_added_since_january",
            "people_added_since_january_with_complete_address",
            "people_added_since_january_with_no_email",
            "people_added_since_january_with_address_but_no_email",
            "people_added_since_january_postcard_candidates",
            "share_of_people_added_since_january",
            "share_of_people_added_since_january_with_complete_address",
        ],
    ),
    "",
    (
        f"Interpretation: {postcard_outreach_since_january_summary['people_added_since_january_postcard_candidates']:,} "
        "people added since January have a complete mailing address, no email on file, and are not flagged "
        f"`nomail`. That is {postcard_outreach_since_january_summary['share_of_people_added_since_january']}% of all "
        "people added since January and "
        f"{postcard_outreach_since_january_summary['share_of_people_added_since_january_with_complete_address']}% of "
        "those with a complete address."
    ),
    "",
    markdown_table(
        postcard_outreach_since_january_rows,
        [
            "vanid",
            "contact_name",
            "address",
            "city",
            "state_province",
            "zip_postal",
            "date_created",
            "origincodename",
        ],
    ),
    "",
    "## Donor Universe vs Leadership Council",
    "",
    markdown_table(
        [donor_vs_leadership_council_summary],
        [
            "total_donors",
            "person_like_donors",
            "total_lc_members",
            "lc_members_in_person_like_donor_base",
            "lc_members_outside_person_like_donor_base",
            "lc_share_of_person_like_donors",
        ],
    ),
    "",
    (
        "Interpretation: the chart below visualizes total donors against Leadership Council members "
        "inferred from activist codes `19LCPin` and `20LCPin`, whose descriptions reference LC pin "
        "mailings. The detailed table also shows that 247 of the 248 LC-coded records fall inside the "
        "person-like donor universe used elsewhere in this analysis."
    ),
    "",
    f"![Donors vs Leadership Council](./figures/{DONOR_VS_LEADERSHIP_COUNCIL_FIGURE_PATH.name})",
    "",
    "## Summary",
    "",
    markdown_table(
        [lapsed_donor_summary],
        [
            "candidate_count",
            "candidate_250plus_count",
            "tier_1_count",
            "tier_2_count",
            "tier_3_count",
            "tier_4_count",
            "avg_giving_years",
            "avg_longest_streak",
            "avg_lifetime_amount",
            "avg_max_annual_amount",
        ],
    ),
    "",
    "## Last Gift Year Mix",
    "",
    markdown_table(lapsed_donor_last_gift_year_rows, ["last_gift_year", "candidate_count", "candidate_250plus_count"]),
    "",
    "## Top Priority Examples",
    "",
    markdown_table(
        lapsed_donor_top_rows,
        [
            "vanid",
            "contact_name",
            "recapture_priority",
            "giving_years",
            "longest_streak",
            "last_gift_date",
            "days_since_last_gift",
            "lifetime_amount",
            "max_annual_amount",
            "years_at_250_plus",
        ],
    ),
    "",
    (
        f"CSV exports: `./{LAPSED_DONOR_EXPORT_PATH.name}`, `./{LAPSED_DONOR_250PLUS_EXPORT_PATH.name}`, "
        f"and `./{POSTCARD_OUTREACH_EXPORT_PATH.name}`"
    ),
]

REPORT_PATH.write_text("\n".join(report_lines), encoding="utf-8")
print(REPORT_PATH)

executive_report_lines = [
    "# Fundraising / Outreach Executive Summary",
    "",
    (
        f"As of {ANALYSIS_AS_OF_DATE}, this report summarizes four practical outreach questions and highlights the "
        "recommended actions more than the underlying technical method."
    ),
    "",
    "## Four Answers at a Glance",
    "",
    markdown_table(executive_summary_rows, ["question", "result", "recommended_action"]),
    "",
    "## 1. How many people are reachable by mail but missing an email address?",
    "",
    (
        f"The current database contains {database_mail_without_email_summary['mailing_address_but_no_email_and_mailable']:,} "
        "`Person` records that have a complete mailing address, no email address on file, and no `nomail` flag. "
        "This is the clearest direct-mail opportunity pool."
    ),
    "",
    f"![Direct-mail opportunity](./figures/{MAILING_NO_EMAIL_FIGURE_PATH.name})",
    "",
    (
        f"These people account for {round(100.0 * database_mail_without_email_summary['mailing_address_but_no_email_and_mailable'] / database_mail_without_email_summary['total_people'], 2)}% "
        "of all `Person` records, which is large enough to justify a dedicated direct-mail, email-capture, or reactivation workflow."
    ),
    "",
    "## 2. Which people added since January are good postcard candidates?",
    "",
    (
        f"Since {POSTCARD_OUTREACH_START_DATE}, the database has added "
        f"{postcard_outreach_since_january_summary['people_added_since_january']:,} `Person` records. "
        f"Of those, {postcard_outreach_since_january_summary['people_added_since_january_postcard_candidates']:,} already meet the criteria for postcard outreach."
    ),
    "",
    f"![Postcard candidates since January](./figures/{POSTCARD_SINCE_JANUARY_FIGURE_PATH.name})",
    "",
    (
        f"Within this group, {postcard_candidates_in_arizona} live in Arizona and {postcard_candidates_outside_arizona} live outside the state. "
        "Because the list is so small, the best next move is to execute a welcome-postcard or light reactivation test immediately rather than expanding the analysis further."
    ),
    "",
    markdown_table(
        postcard_outreach_since_january_rows,
        ["vanid", "contact_name", "city", "state_province", "date_created", "origincodename"],
    ),
    "",
    "## 3. How large is Leadership Council within the donor base?",
    "",
    (
        f"The inferred Leadership Council population is {donor_vs_leadership_council_summary['total_lc_members']:,}. "
        f"Of those, {donor_vs_leadership_council_summary['lc_members_in_person_like_donor_base']:,} fall inside the current donor base, "
        f"representing just {donor_vs_leadership_council_summary['lc_share_of_person_like_donors']}% of person-like donors."
    ),
    "",
    f"![Donors vs Leadership Council](./figures/{DONOR_VS_LEADERSHIP_COUNCIL_FIGURE_PATH.name})",
    "",
    "This shows that LC is an important but very small high-value segment, better handled as its own stewardship, upgrade, or special-care audience.",
    "",
    "## 4. Which lapsed donors should be prioritized for reactivation?",
    "",
    (
        f"The analysis identified {lapsed_donor_summary['candidate_count']:,} consistent-but-lapsed donors. "
        f"Among them, {tier_1_and_2_count:,} fall into the high-priority tiers, and "
        f"{lapsed_donor_summary['candidate_250plus_count']:,} have at least one year of `$250+` giving."
    ),
    "",
    f"![Lapsed donor priority tiers](./figures/{LAPSED_DONOR_PRIORITY_FIGURE_PATH.name})",
    "",
    (
        f"The biggest concentration of recent lapse sits in 2024-2025, with {recent_lapsed_2024_2025_count:,} people across those two years. "
        "The most promising reactivation audience is not the oldest lapse group, but the people who stopped giving recently after a more stable giving history."
    ),
    "",
    markdown_table(
        [lapsed_donor_summary],
        [
            "candidate_count",
            "candidate_250plus_count",
            "tier_1_count",
            "tier_2_count",
            "tier_3_count",
            "tier_4_count",
        ],
    ),
    "",
    "## Recommended Next Steps",
    "",
    "1. Build a dedicated outreach segment for the `16,438` mail-reachable people who have no email address.",
    f"2. Contact the 6 people in `./{POSTCARD_OUTREACH_EXPORT_PATH.name}` and test a welcome or reactivation postcard.",
    f"3. Run Leadership Council as its own stewardship cadence for `248` people instead of mixing it into general donor cultivation.",
    f"4. Start with Tier 1 and Tier 2 in `./{LAPSED_DONOR_250PLUS_EXPORT_PATH.name}` and `./{LAPSED_DONOR_EXPORT_PATH.name}`, covering {tier_1_and_2_count} people.",
    "",
    "## Supporting Files",
    "",
    f"- Technical report: `./{REPORT_PATH.name}`",
    f"- Executive summary: `./{EXECUTIVE_SUMMARY_REPORT_PATH.name}`",
    f"- Postcard list: `./{POSTCARD_OUTREACH_EXPORT_PATH.name}`",
    f"- Lapsed donor full list: `./{LAPSED_DONOR_EXPORT_PATH.name}`",
    f"- Lapsed donor $250+ list: `./{LAPSED_DONOR_250PLUS_EXPORT_PATH.name}`",
]

EXECUTIVE_SUMMARY_REPORT_PATH.write_text("\n".join(executive_report_lines), encoding="utf-8")
print(EXECUTIVE_SUMMARY_REPORT_PATH)

executive_notebook = nbf.v4.new_notebook()
executive_notebook.cells = [
    nbf.v4.new_markdown_cell(
        "# Fundraising / Outreach Executive Summary\n\n"
        f"As of {ANALYSIS_AS_OF_DATE}, this notebook answers four practical outreach questions in an executive-ready format."
    ),
    nbf.v4.new_code_cell(
        "from pathlib import Path\n\n"
        f"ANALYSIS_AS_OF_DATE = '{ANALYSIS_AS_OF_DATE}'\n"
        f"PROJECT_DIR = Path(r'{PROJECT_DIR}')\n"
        "PROJECT_DIR"
    ),
    nbf.v4.new_markdown_cell("## Four Answers at a Glance"),
    nbf.v4.new_code_cell(
        "# Executive summary table",
        outputs=[
            nbf.v4.new_output(
                "display_data",
                data={"text/markdown": markdown_table(executive_summary_rows, ["question", "result", "recommended_action"])},
            )
        ],
    ),
    nbf.v4.new_markdown_cell(
        "## 1. How many people are reachable by mail but missing an email address?\n\n"
        f"The database contains **{database_mail_without_email_summary['mailing_address_but_no_email_and_mailable']:,}** people "
        "who have a complete mailing address, no email on file, and no `nomail` flag. "
        "This is the clearest direct-mail opportunity pool."
    ),
    nbf.v4.new_markdown_cell(f"![Direct-mail opportunity](./figures/{MAILING_NO_EMAIL_FIGURE_PATH.name})"),
    nbf.v4.new_code_cell(
        "mail_reachable_without_email = {\n"
        f"    'total_people': {database_mail_without_email_summary['total_people']},\n"
        f"    'people_with_complete_mailing_address': {database_mail_without_email_summary['people_with_complete_mailing_address']},\n"
        f"    'people_with_no_email': {database_mail_without_email_summary['people_with_no_email']},\n"
        f"    'mailing_address_but_no_email_and_mailable': {database_mail_without_email_summary['mailing_address_but_no_email_and_mailable']}\n"
        "}\n"
        "mail_reachable_without_email",
        outputs=[
            nbf.v4.new_output(
                "execute_result",
                data={"text/plain": repr(
                    {
                        "total_people": database_mail_without_email_summary["total_people"],
                        "people_with_complete_mailing_address": database_mail_without_email_summary["people_with_complete_mailing_address"],
                        "people_with_no_email": database_mail_without_email_summary["people_with_no_email"],
                        "mailing_address_but_no_email_and_mailable": database_mail_without_email_summary["mailing_address_but_no_email_and_mailable"],
                    }
                )},
                execution_count=1,
            )
        ],
        execution_count=1,
    ),
    nbf.v4.new_markdown_cell(
        "## 2. Which people added since January are good postcard candidates?\n\n"
        f"Since **{POSTCARD_OUTREACH_START_DATE}**, the database has added "
        f"**{postcard_outreach_since_january_summary['people_added_since_january']:,}** people. "
        f"Only **{postcard_outreach_since_january_summary['people_added_since_january_postcard_candidates']:,}** currently qualify for postcard outreach."
    ),
    nbf.v4.new_markdown_cell(f"![Postcard candidates since January](./figures/{POSTCARD_SINCE_JANUARY_FIGURE_PATH.name})"),
    nbf.v4.new_code_cell(
        "# Postcard candidate list",
        outputs=[
            nbf.v4.new_output(
                "display_data",
                data={
                    "text/markdown": markdown_table(
                        postcard_outreach_since_january_rows,
                        ["vanid", "contact_name", "city", "state_province", "date_created", "origincodename"],
                    )
                },
            )
        ],
    ),
    nbf.v4.new_markdown_cell(
        "## 3. How large is Leadership Council within the donor base?\n\n"
        f"Leadership Council includes **{donor_vs_leadership_council_summary['total_lc_members']:,}** people, and "
        f"**{donor_vs_leadership_council_summary['lc_members_in_person_like_donor_base']:,}** of them are inside the donor base used in this analysis. "
        f"That equals **{donor_vs_leadership_council_summary['lc_share_of_person_like_donors']}%** of person-like donors."
    ),
    nbf.v4.new_markdown_cell(f"![Donors vs Leadership Council](./figures/{DONOR_VS_LEADERSHIP_COUNCIL_FIGURE_PATH.name})"),
    nbf.v4.new_markdown_cell(
        "## 4. Which lapsed donors should be prioritized for reactivation?\n\n"
        f"The analysis identified **{lapsed_donor_summary['candidate_count']:,}** consistent-but-lapsed donors. "
        f"Among them, **{tier_1_and_2_count:,}** are high-priority and **{lapsed_donor_summary['candidate_250plus_count']:,}** have a `$250+` giving history."
    ),
    nbf.v4.new_markdown_cell(f"![Lapsed donor priority tiers](./figures/{LAPSED_DONOR_PRIORITY_FIGURE_PATH.name})"),
    nbf.v4.new_code_cell(
        "# Lapsed donor priority table",
        outputs=[
            nbf.v4.new_output(
                "display_data",
                data={
                    "text/markdown": markdown_table(
                        [lapsed_donor_summary],
                        [
                            "candidate_count",
                            "candidate_250plus_count",
                            "tier_1_count",
                            "tier_2_count",
                            "tier_3_count",
                            "tier_4_count",
                        ],
                    )
                },
            )
        ],
    ),
    nbf.v4.new_markdown_cell(
        "## Recommended Next Steps\n\n"
        "1. Build a dedicated outreach segment for the mail-reachable people who have no email address.\n"
        f"2. Use `./{POSTCARD_OUTREACH_EXPORT_PATH.name}` for an immediate postcard test.\n"
        "3. Treat Leadership Council as a separate stewardship audience.\n"
        f"4. Start reactivation with Tier 1 and Tier 2 from `./{LAPSED_DONOR_250PLUS_EXPORT_PATH.name}` and `./{LAPSED_DONOR_EXPORT_PATH.name}`."
    ),
]
executive_notebook.metadata = {
    "kernelspec": {
        "display_name": "Python 3",
        "language": "python",
        "name": "python3",
    },
    "language_info": {
        "name": "python",
        "version": "3",
    },
}
EXECUTIVE_SUMMARY_NOTEBOOK_PATH.write_text(nbf.writes(executive_notebook), encoding="utf-8")
print(EXECUTIVE_SUMMARY_NOTEBOOK_PATH)

conn.close()
